# Trimming della coda 3′
Taglia la coda 3′ di R1 e di R2 alla prima base, letta dalla fine, con qualità almeno pari alla soglia
scelta dall'utente (`q_min`).

Il file contiene tre funzioni:
- `phred_scores`: converte la riga di qualità da caratteri ASCII a punteggi numerici;
- `trim_read`: taglia la coda di una sola read e restituisce la tupla `(sequenza, qualità)`;
- `trim_3prime`: applica `trim_read` alle due read di una coppia e restituisce le tuple
  `trimmed_forward` (R1) e `trimmed_reverse` (R2), ciascuna con sequenza e qualità.

Nel flusso del programma sta fra la validazione dei file (`leggi_e_valida_coppie`, che consegna le coppie
di read) e il controllo di lunghezza (`check_Length`, che riceve le due tuple tagliate).

In [ ]:
def phred_scores(quality, offset=33):
    """I punteggi di qualità Phred di una riga di qualità FASTQ, uno per base.

    Ogni carattere della riga codifica un numero: il suo codice ASCII meno 33
    (codifica Phred+33, quella di Illumina). "!" vale 0, "5" vale 20, "I" vale 40.
    Il punteggio Q corrisponde a una probabilità di errore di 10^(-Q/10):
    Q = 20 significa un errore ogni 100 basi.

    **Perché ord.** In Python un carattere è una stringa lunga 1, non un numero:
    ord() ne restituisce il codice ASCII, su cui si può fare la sottrazione.
    """
    scores = []
    for char in quality:
        scores.append(ord(char) - offset)
    return scores


In [ ]:
def trim_read(sequence, quality, q_min=20):
    """Una read con la coda 3' di bassa qualità tagliata: la tupla (sequenza, qualità).

    La read si legge dalla fine verso l'inizio (slicing [::-1]) e si contano
    le basi con qualità sotto q_min, fermandosi alla prima base che raggiunge
    la soglia: quella e tutte le precedenti si tengono. Il taglio è uguale per
    la sequenza e per la riga di qualità, così le due restano allineate.

    **Solo la coda.** Una base di bassa qualità in mezzo alla read non viene
    toccata: il trimming 3' rimuove soltanto le basi scadenti all'estremità,
    dove la qualità del sequenziamento Illumina tende a calare.

    **La soglia.** Una base si tiene se la sua qualità è maggiore o uguale a
    q_min. Il valore predefinito è 20, quello delle slide (Qmin = 20).

    **Casi limite.** Se nessuna base della coda è sotto soglia la read resta
    intera; se lo sono tutte, sequenza e qualità diventano stringhe vuote, e
    sarà il controllo di lunghezza a scartare la read.

    **Chi la usa.** ``trim_3prime``, una volta per ciascuna delle due read della coppia.
    """
    # Si tolgono eventuali "\n" o spazi ai lati, nel caso le righe arrivino
    # dal file senza essere state pulite: un "\n" verrebbe letto come una base.
    sequence = sequence.strip()
    quality = quality.strip()

    scores = phred_scores(quality)

    # Quante basi togliere dalla fine: si scorrono i punteggi al contrario
    # e ci si ferma alla prima base che raggiunge la soglia.
    n_cut = 0
    for score in scores[::-1]:
        if score >= q_min:
            break
        n_cut += 1

    # Si taglia fino a "lunghezza - basi da togliere". Si evita sequence[:-n_cut]
    # perché con n_cut = 0 diventerebbe sequence[:-0] = sequence[:0], cioè
    # una stringa vuota: una read perfetta verrebbe cancellata invece che tenuta.
    end = len(sequence) - n_cut
    trimmed_read = (sequence[:end], quality[:end])  # tupla: (sequenza tagliata, qualità tagliata)
    return trimmed_read


In [ ]:
def trim_3prime(forward, reverse, q_min=20):
    """Le due read di una coppia con la coda 3' tagliata: (trimmed_forward, trimmed_reverse).

    **Cosa riceve.** ``forward`` (R1) e ``reverse`` (R2) sono le due read della
    coppia come le consegna la validazione dei file (``leggi_e_valida_coppie``):
    ciascuna è una tupla di quattro stringhe (intestazione, sequenza, riga
    ``+``, qualità). Qui servono solo la sequenza (indice 1) e la qualità (indice 3).

    **Cosa restituisce.** ``trimmed_forward`` e ``trimmed_reverse``, separate:
    ciascuna è una tupla di due stringhe (sequenza, qualità), nel formato che
    ``check_Length`` si aspetta. L'intestazione e la riga ``+`` non vengono
    toccate né restituite: restano nelle tuple originali, e servono a
    ``write_fastq`` per riscrivere il record.

    **Una read alla volta, due soglie uguali.** ``q_min`` è la stessa per R1 e
    per R2, ma ogni read si taglia per conto suo: dopo il taglio possono avere
    lunghezze diverse, ed è normale. Sarà il controllo di lunghezza a decidere
    sulla coppia.

    **Chi la usa.** Il programma principale, una volta per ogni coppia, dopo la
    validazione e prima del controllo di lunghezza. ``q_min`` deve essere un
    intero, come lo restituisce ``acquisisci_soglie``:
    trimmed_forward, trimmed_reverse = trim_3prime(forward, reverse, q_min)
    """
    trimmed_forward = trim_read(forward[1], forward[3], q_min)
    trimmed_reverse = trim_read(reverse[1], reverse[3], q_min)
    return trimmed_forward, trimmed_reverse
